# Intuition test: regression

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Seeded and self-contained.

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from common.datasets import make_noisy_curve, make_sparse_regression

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

## Q1. Degree against error

You fit polynomials of degree 1, 3, 6 and 15 to 30 noisy points of a smooth curve, then score each on 300 fresh points. What happens to the training error as the degree grows? What happens to the error on the fresh points? At roughly which degree would you expect the fresh error to be lowest, and where does the training error end up?

Prediction: *(write here)*

In [ ]:
X_tr, y_tr = make_noisy_curve(n_samples=30, noise=0.3, seed=1)
X_te, y_te = make_noisy_curve(n_samples=300, noise=0.3, seed=99)
for d in (1, 3, 6, 15):
    m = make_pipeline(
        PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()
    )
    m.fit(X_tr, y_tr)
    print(
        f"degree {d:>2}: train {mean_squared_error(y_tr, m.predict(X_tr)):8.3f}  fresh {mean_squared_error(y_te, m.predict(X_te)):10.3f}"
    )

## Q2. Which regulariser

You have 100 rows and 50 features. You believe only about 5 matter, but you do not know which. Ridge, Lasso or plain least squares? Which gives the lowest test error, and which would you hand to a colleague who wants to know what drives the outcome?

Prediction: *(write here)*

In [ ]:
Xa, ya, w_true = make_sparse_regression(
    n_samples=500, n_features=50, n_informative=5, noise=3.0, seed=SEED
)
Xtr, Xte, ytr, yte = Xa.iloc[:100], Xa.iloc[100:], ya.iloc[:100], ya.iloc[100:]
for name, m in [
    ("OLS", LinearRegression()),
    ("Ridge(10)", Ridge(alpha=10)),
    ("Lasso(0.3)", Lasso(alpha=0.3)),
]:
    m.fit(Xtr, ytr)
    kept = int(np.sum(np.abs(m.coef_) > 1e-8))
    print(
        f"{name:>10}: test MSE {mean_squared_error(yte, m.predict(Xte)):6.2f}, non-zero weights {kept}"
    )

## Q3. A learning rate that is too high

You run gradient descent on a bowl-shaped cost, and the learning rate is a bit above the stability limit. Sketch what the cost curve looks like over the first 20 steps. Does it plateau? Does it wobble and settle? Now halve the rate and sketch it again.

Prediction: *(write here)*

In [ ]:
x = rng.normal(size=200)
y = 3 * x + rng.normal(0, 0.5, 200)
limit = 2 / (2 * np.mean(x**2))  # stability limit for the one-weight cost
for lr in (1.1 * limit, 0.55 * limit):
    w, costs = 0.0, []
    for _ in range(20):
        costs.append(float(np.mean((y - w * x) ** 2)))
        w += lr * 2 * np.mean(x * (y - w * x))
    print(f"lr {lr:.2f}:", np.round(costs[:8], 2), "... step 20:", f"{costs[-1]:.3g}")

## Q4. Scaling

One feature is the number of bedrooms (1 to 5). Another is the floor area in square centimetres (about 1,000,000). You run gradient descent with a single learning rate. Which feature forces the rate to be tiny, and what happens to the weight on bedrooms?

Prediction: *(write here)*

In [ ]:
beds = rng.integers(1, 6, 300).astype(float)
area = rng.normal(1_000_000, 200_000, 300)
price = 50_000 * beds + 0.05 * area + rng.normal(0, 20_000, 300)
A = np.column_stack([beds, area])
print("column scales:", A.std(axis=0))
print("Hessian eigenvalues:", np.linalg.eigvalsh(2 / len(A) * (A - A.mean(0)).T @ (A - A.mean(0))))
Z = StandardScaler().fit_transform(A)
print("after scaling:", np.linalg.eigvalsh(2 / len(Z) * Z.T @ Z))

## Q5. A duplicated column

You fit least squares on one feature $x$ and get a weight of 2.0. Now you add an exact copy of $x$ as a second column and refit. What happens to the two weights? What happens to the predictions? Then repeat with Ridge.

Prediction: *(write here)*

In [ ]:
x = rng.normal(size=100)
y = 2 * x + rng.normal(0, 0.5, 100)
both = np.column_stack([x, x])
print("OLS  one column  :", LinearRegression().fit(x.reshape(-1, 1), y).coef_)
ols2 = LinearRegression().fit(both, y)
print(
    "OLS  two columns :",
    ols2.coef_,
    " prediction gap:",
    np.abs(
        ols2.predict(both) - LinearRegression().fit(x.reshape(-1, 1), y).predict(x.reshape(-1, 1))
    ).max(),
)
print("Ridge two columns:", Ridge(alpha=1.0).fit(both, y).coef_)

## Q6. Leakage

You have 100 rows, 500 random features and a target that is pure noise. You pick the 10 features most correlated with the target, then run 5-fold cross-validation on those 10. What cross-validated $R^2$ do you expect: about 0, clearly positive or clearly negative? What if the selection happens inside the pipeline instead?

Prediction: *(write here)*

In [ ]:
Xn = rng.normal(size=(100, 500))
yn = rng.normal(size=100)
cv = KFold(5, shuffle=True, random_state=SEED)
outside = cross_val_score(
    LinearRegression(),
    SelectKBest(f_regression, k=10).fit_transform(Xn, yn),
    yn,
    cv=cv,
    scoring="r2",
)
inside = cross_val_score(
    make_pipeline(SelectKBest(f_regression, k=10), LinearRegression()), Xn, yn, cv=cv, scoring="r2"
)
print("selection outside CV:", outside.mean())
print("selection inside CV :", inside.mean())

## Q7. The extremes of the penalty

For Ridge, what happens to the weights as $\alpha\to0$, and as $\alpha\to\infty$? What does the model predict in the second case?

Prediction: *(write here)*

In [ ]:
Xs = StandardScaler().fit_transform(Xtr)
for a in (1e-6, 10, 1e3, 1e6):
    m = Ridge(alpha=a).fit(Xs, ytr)
    print(
        f"alpha {a:>9.0e}: largest |weight| {np.abs(m.coef_).max():8.3f}   prediction spread (sd) {m.predict(Xs).std():.3f}"
    )
print("sd of y:", ytr.std())

---

## Answers

**Q1.** Training error falls at every step and ends close to zero at degree 15 (it can even dip below the noise level of 0.09). Error on fresh data falls to a minimum around degree 6 (about 0.10, close to the 0.09 noise floor) and then explodes to the hundreds at degree 15, because the curve swings wildly between the points. Underfitting on the left, overfitting on the right. Only the fresh-data error tells you which side you are on.

**Q2.** Lasso. With 5 real features in 50, it zeros most of the noise columns, gives the lowest test error here and leaves a short list to explain. Ridge improves on least squares but keeps all 50 weights, and OLS overfits. If every feature mattered a little, Ridge would be the better pick.

**Q3.** Above the limit, each step overshoots the minimum by more than it started away from it. The cost grows from step to step, with no plateau and no settling. It is 7 at the start and about 7,000 after 20 steps. Halving the rate (to 0.55 times the limit) falls to the minimum almost at once, in one step for this single-weight problem. A cost curve that rises is the signal to cut the rate.

**Q4.** The floor-area column. The largest Hessian eigenvalue is about $8\times10^{10}$ against 4 for bedrooms, and the stable rate must sit below $2/\lambda_{\max}$, so the rate is tiny. The bedroom weight then moves very slowly, because its own gradient is small relative to that rate. Standardising puts both eigenvalues near 2.

**Q5.** The two weights split the credit, about 1.0 each for OLS, with identical predictions. (Only their sum is determined, and the solver picks the smallest-norm split.) Ridge also splits evenly, at slightly under 1.0 each, because the penalty prefers spreading weight across the columns to loading it on one. With near-copies instead of exact copies, the split becomes unstable from one sample to the next.

**Q6.** The leaky version comes out positive (about 0.1 here), a real-looking model built from pure noise, because the selection step has already seen the labels. Inside the pipeline the selection is refitted on each training fold and the score is negative (about -0.7 here), worse than predicting the mean, which is the honest verdict on noise. When a score looks too good, check for leakage first.

**Q7.** As $\alpha\to0$ Ridge becomes ordinary least squares, with large weights that overfit. As $\alpha\to\infty$ every weight is pushed to zero and the model predicts the training mean for every row, with a prediction spread of zero. The useful values are in between, and cross-validation picks them.